In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [10]:
!pip install dagshub mlflow -q

# Decision Tree

## Setup And Imports

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='fraud-detection', mlflow=True)
mlflow.set_experiment('DecisionTree')

RANDOM_STATE = 42
print('done')

Initialized MLflow to track repo "LukaBatilashvili07/fraud-detection"

Repository LukaBatilashvili07/fraud-detection initialized!

2026/05/03 08:16:05 INFO mlflow.tracking.fluent: Experiment with name 'DecisionTree' does not exist. Creating a new experiment.


done


## Data Loading

In [13]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
train.head()

Train shape: (590540, 434)
Test shape: (506691, 433)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## Cleaning

In [15]:
with mlflow.start_run(run_name='DecisionTree_Cleaning'):
    y = train['isFraud']
    X = train.drop(['isFraud', 'TransactionID'], axis=1)
    test_ids = test['TransactionID']
    X_test = test.drop(['TransactionID'], axis=1)
    null_ratio = X.isnull().mean()
    drop_cols = null_ratio[null_ratio > 0.5].index
    X = X.drop(columns=drop_cols)
    X_test = X_test.drop(columns=drop_cols, errors='ignore')

    cat_cols = X.select_dtypes(include=['object']).columns
    le = LabelEncoder()
    for col in cat_cols:
        X[col] = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)
        le.fit(pd.concat([X[col], X_test[col]]))
        X[col] = le.transform(X[col])
        X_test[col] = le.transform(X_test[col])

    medians = X.median()
    X = X.fillna(medians)
    X_test = X_test.fillna(medians)

    mlflow.log_param('dropped_columns_count', len(drop_cols))
    mlflow.log_param('missing_fill_strategy', 'median')
    mlflow.log_param('categorical_encoding', 'LabelEncoder')
    mlflow.log_metric('train_rows', len(X))
    mlflow.log_metric('train_cols', len(X.columns))
    print('Shape after cleaning:', X.shape)

Shape after cleaning: (590540, 218)
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/d9cc655a48bd4cb289290299e051ed7e
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3


## Feature Engineering

In [17]:
with mlflow.start_run(run_name="DecisionTree_Feature_Engineering"):
    def add_features(df):
        df = df.copy()
        df['amt_log'] = np.log1p(df['TransactionAmt'])
        df['amt_cents'] = df['TransactionAmt'] % 1
        return df

    X = add_features(X)
    X_test = add_features(X_test)

    X = X.fillna(X.median())
    X_test = X_test.fillna(X.median())

    mlflow.log_param('added_features', 'amt_log, amt_cents')
    mlflow.log_metric('feature_count', X.shape[1])

    print('Shape:', X.shape)

Shape: (590540, 220)
🏃 View run DecisionTree_Feature_Engineering at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/127243b4b05740dbbc2328214deacd7b
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3


## Feature Selection

In [19]:
with mlflow.start_run(run_name='DecisionTree_Feature_Selection'):
    kbest = SelectKBest(f_classif, k=50)
    kbest.fit(X, y)
    kbest_cols = X.columns[kbest.get_support()]

    selected_cols = kbest_cols
    X_selected = X[selected_cols]
    X_test_selected = X_test[selected_cols]

    mlflow.log_param('selection_method', 'SelectKBest_f_classif')
    mlflow.log_param('k', 50)
    mlflow.log_metric('final_feature_count', len(selected_cols))

    print('Final features:', len(selected_cols))

Final features: 50
🏃 View run DecisionTree_Feature_Selection at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/279ea2428b2e452bb7b06f0247c860e5
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3


## Training

In [20]:
X_train, X_val, y_train, y_val = train_test_split(
    X_selected, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

def get_scores(model, X_train, y_train, X_val, y_val):
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    return train_auc, val_auc

results = {}

with mlflow.start_run(run_name='DecisionTree_NoLimit'):
    pipe = Pipeline([
        ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ])
    pipe.fit(X_train, y_train)
    train_auc, val_auc = get_scores(pipe, X_train, y_train, X_val, y_val)
    mlflow.log_param('max_depth', 'None')
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('val_auc', val_auc)
    mlflow.sklearn.log_model(pipe, 'model')
    results['DT No Limit'] = {'train_auc': train_auc, 'val_auc': val_auc}
    print(f'DT - Train AUC: {train_auc:.4f}, Val AUC: {val_auc:.4f}')

2026/05/03 08:47:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 08:47:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


DT - Train AUC: 0.7768, Val AUC: 0.7313
🏃 View run DecisionTree_NoLimit at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/467eb7654c43460299206f7454d8d569
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3


In [21]:
with mlflow.start_run(run_name='DecisionTree_MaxDepth5'):
    pipe_d5 = Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE))
    ])
    pipe_d5.fit(X_train, y_train)
    train_auc, val_auc = get_scores(pipe_d5, X_train, y_train, X_val, y_val)
    mlflow.log_param('max_depth', 5)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('val_auc', val_auc)
    mlflow.sklearn.log_model(pipe_d5, 'model')
    results['DT max depth 5'] = {'train_auc': train_auc, 'val_auc': val_auc}
    print(f'DT max depth 5 - Train AUC: {train_auc:.4f}, Val AUC: {val_auc:.4f}')

2026/05/03 08:50:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 08:50:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


DT max depth 5 - Train AUC: 0.7050, Val AUC: 0.7091
🏃 View run DecisionTree_MaxDepth5 at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/e48b600b50e347a2be4180bb4fd848e6
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3


In [23]:
with mlflow.start_run(run_name='DecisionTree_MaxDepth10'):
    pipe_d10 = Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE))
    ])
    pipe_d10.fit(X_train, y_train)
    train_auc, val_auc = get_scores(pipe_d10, X_train, y_train, X_val, y_val)
    mlflow.log_param('max_depth', 10)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('val_auc', val_auc)
    mlflow.sklearn.log_model(pipe_d10, 'model')
    results['DT max depth 10'] = {'train_auc': train_auc, 'val_auc': val_auc}
    print(f'DT max depth 10 - Train AUC: {train_auc:.4f}, Val AUC: {val_auc:.4f}')

2026/05/03 08:51:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 08:51:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


DT max depth 10 - Train AUC: 0.7370, Val AUC: 0.7357
🏃 View run DecisionTree_MaxDepth10 at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3/runs/a3f7bbaebd534137afe6855b7ca35339
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/fraud-detection.mlflow/#/experiments/3
